# Process Nov 5, 2024 presidential election data.

In [ ]:
import json
import pandas as pd

In [ ]:
# Try to load 2024 presidential data
try:
    with open("2024.president.json", "r") as fp:
        results = json.load(fp)
    print("Loaded 2024 presidential data from JSON file")
except FileNotFoundError:
    print("2024.president.json not found. Please run the data downloader first.")
    results = None

In [ ]:
if results:
    # Check the structure of the results
    print("Data structure:")
    if isinstance(results, dict):
        print("Results is a dictionary with keys:", list(results.keys()))
    elif isinstance(results, list):
        print("Results is a list with", len(results), "items")
        if len(results) > 0:
            print("First item keys:", list(results[0].keys()) if isinstance(results[0], dict) else "Not a dict")
    
    # Function to convert race results to DataFrame
    def race_to_df(race, col_prefix):
        data = list()
        
        # Handle different possible data structures
        if "reporting_units" in race:
            # 2022-style structure
            for county in race["reporting_units"]:
                if county["level"] == "state":
                    continue
                
                row = dict()
                row["county"] = county["name"]
                row[f"{col_prefix}_total"] = county["total_votes"]
                
                total = 0
                for candidate in county["candidates"]:
                    key = f'{col_prefix}_{candidate["nyt_id"].split("-")[0]}'
                    row[key] = candidate["votes"]["total"]
                    total += row[key]
                
                row[f"{col_prefix}_total"] = total
                for candidate in county["candidates"]:
                    key = f'{col_prefix}_{candidate["nyt_id"].split("-")[0]}_pct'
                    row[key] = candidate["votes"]["total"] / total
                
                data.append(row)
                
        elif "counties" in race:
            # 2016-style structure
            for county in race["counties"]:
                row = dict()
                row["county"] = county["name"]
                
                total = 0
                for key, value in county["results"].items():
                    row[f'{col_prefix}_{key.split("-")[0]}'] = value
                    total += value
                
                row[f"{col_prefix}_total"] = total
                for key, value in county["results"].items():
                    row[f'{col_prefix}_{key.split("-")[0]}_pct'] = value / total
                
                data.append(row)
        
        return pd.DataFrame(data)
    
    # Process the presidential race
    if isinstance(results, list) and len(results) > 0:
        # Assume first item is presidential race
        pres_race = results[0]
        df_pres = race_to_df(pres_race, "pres_24")
        print(f"Processed presidential data: {len(df_pres)} counties")
        print(df_pres.head())
        
        # Save to feather format
        df_pres.to_feather("2024.ft")
        print("Saved 2024 data to 2024.ft")
        
    elif isinstance(results, dict):
        # Single race result
        df_pres = race_to_df(results, "pres_24")
        print(f"Processed presidential data: {len(df_pres)} counties")
        print("Saved 2024 data to 2024.ft")
        
    else:
        print("Unexpected data structure. Please check the JSON format.")